In [1]:
import numpy as np

# 1. GRAM-SCHMIDT CLASSIQUE (CGS)
# Objectif : orthonormaliser les colonnes d'une matrice
# Produit une décomposition A = Q R

def cgs(A):
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))

    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            # projection de A[:,j] sur Q[:,i]
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v -= R[i, j] * Q[:, i]

        # normalisation
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]

    return Q, R


# 2. GRAM-SCHMIDT MODIFIÉ (MGS)
# Objectif : version plus stable numériquement de CGS

def mgs(A):
    m, n = A.shape
    Q = A.copy().astype(float)
    R = np.zeros((n, n))

    for i in range(n):
        R[i, i] = np.linalg.norm(Q[:, i])
        Q[:, i] /= R[i, i]

        for j in range(i + 1, n):
            R[i, j] = np.dot(Q[:, i], Q[:, j])
            Q[:, j] -= R[i, j] * Q[:, i]

    return Q, R



def householder_qr(A):
    A = A.copy().astype(float)
    m, n = A.shape
    Q = np.eye(m)

    for k in range(n):
        x = A[k:, k]

        # vecteur de Householder
        e1 = np.zeros_like(x)
        e1[0] = 1

        alpha = np.linalg.norm(x)
        v = x + np.sign(x[0]) * alpha * e1
        v = v / np.linalg.norm(v)

        # matrice de réflexion
        H = np.eye(m)
        H[k:, k:] -= 2 * np.outer(v, v)

        A = H @ A
        Q = Q @ H.T

    return Q, A  # A devient R


# 4. GIVENS
# Objectif : annuler les coefficients un par un avec rotations

def givens_qr(A):
    A = A.copy().astype(float)
    m, n = A.shape
    Q = np.eye(m)

    for j in range(n):
        for i in range(m - 1, j, -1):
            a, b = A[i - 1, j], A[i, j]

            if b == 0:
                continue

            r = np.hypot(a, b)
            c = a / r
            s = -b / r

            G = np.eye(m)
            G[i - 1, i - 1] = c
            G[i, i] = c
            G[i - 1, i] = -s
            G[i, i - 1] = s

            A = G @ A
            Q = Q @ G.T

    return Q, A


# 5. ALGORITHME QR
# Objectif : calculer les valeurs propres

def qr_algorithm(A, iterations=100):
    A = A.copy().astype(float)

    for _ in range(iterations):
        Q, R = np.linalg.qr(A)
        A = R @ Q

    return np.diag(A)


# 6. MÉTHODE DE JACOBI
# Objectif : valeurs propres pour matrices symétriques

def jacobi(A, tol=1e-10, max_iter=100):
    A = A.copy().astype(float)
    n = A.shape[0]

    for _ in range(max_iter):
        # chercher le plus grand élément hors diagonale
        off_diag = np.abs(A - np.diag(np.diag(A)))
        p, q = np.unravel_index(np.argmax(off_diag), A.shape)

        if abs(A[p, q]) < tol:
            break

        theta = 0.5 * np.arctan2(2*A[p, q], A[q, q] - A[p, p])
        c = np.cos(theta)
        s = np.sin(theta)

        J = np.eye(n)
        J[p, p] = c
        J[q, q] = c
        J[p, q] = s
        J[q, p] = -s

        A = J.T @ A @ J

    return np.diag(A)


# 7. MÉTHODE DE DAVIDSON (simplifiée)
# Objectif : trouver quelques valeurs propres (grandes matrices)

def davidson(A, max_iter=100):
    n = A.shape[0]
    v = np.random.rand(n, 1)
    v, _ = np.linalg.qr(v)

    for _ in range(max_iter):
        Av = A @ v
        T = v.T @ Av

        eigvals, eigvecs = np.linalg.eig(T)
        theta = eigvals[0]
        u = v @ eigvecs[:, 0]

        r = A @ u - theta * u

        if np.linalg.norm(r) < 1e-8:
            return theta

        # correction
        t = r / (np.diag(A) - theta + 1e-10)
        v = np.column_stack((v, t))
        v, _ = np.linalg.qr(v)

    return theta




In [2]:
# MAIN 

if __name__ == "__main__":

    # Matrice de test
    A = np.array([[4, 1, 2],
                  [1, 3, 0],
                  [2, 0, 1]], dtype=float)

    print("Matrice A :\n", A)

    # --- Gram-Schmidt
    Q, R = cgs(A)
    print("\nCGS -> Q:\n", Q)
    print("CGS -> R:\n", R)

    Q, R = mgs(A)
    print("\nMGS -> Q:\n", Q)
    print("MGS -> R:\n", R)

    # --- Householder
    Q, R = householder_qr(A)
    print("\nHouseholder -> Q:\n", Q)
    print("Householder -> R:\n", R)

    # --- Givens
    Q, R = givens_qr(A)
    print("\nGivens -> Q:\n", Q)
    print("Givens -> R:\n", R)

    # --- QR Algorithm (valeurs propres)
    eig_qr = qr_algorithm(A)
    print("\nValeurs propres (QR) :", eig_qr)

    # --- Jacobi
    eig_jacobi = jacobi(A)
    print("Valeurs propres (Jacobi) :", eig_jacobi)

    # --- Davidson
    eig_davidson = davidson(A)
    print("Plus petite valeur propre (Davidson) :", eig_davidson)

Matrice A :
 [[4. 1. 2.]
 [1. 3. 0.]
 [2. 0. 1.]]

CGS -> Q:
 [[ 0.87287156 -0.12038585  0.47286624]
 [ 0.21821789  0.96308682 -0.15762208]
 [ 0.43643578 -0.24077171 -0.86692145]]
CGS -> R:
 [[ 4.58257569  1.52752523  2.1821789 ]
 [ 0.          2.76887462 -0.48154341]
 [ 0.          0.          0.07881104]]

MGS -> Q:
 [[ 0.87287156 -0.12038585  0.47286624]
 [ 0.21821789  0.96308682 -0.15762208]
 [ 0.43643578 -0.24077171 -0.86692145]]
MGS -> R:
 [[ 4.58257569  1.52752523  2.1821789 ]
 [ 0.          2.76887462 -0.48154341]
 [ 0.          0.          0.07881104]]

Householder -> Q:
 [[-0.87287156  0.12038585  0.47286624]
 [-0.21821789 -0.96308682 -0.15762208]
 [-0.43643578  0.24077171 -0.86692145]]
Householder -> R:
 [[-4.58257569e+00 -1.52752523e+00 -2.18217890e+00]
 [ 3.41281319e-17 -2.76887462e+00  4.81543412e-01]
 [ 2.34675148e-16  2.45279335e-17  7.88110406e-02]]

Givens -> Q:
 [[ 0.87287156 -0.12038585 -0.47286624]
 [ 0.21821789  0.96308682  0.15762208]
 [ 0.43643578 -0.24077171  0